# Primer: enough Python to begin

Work through this primer before Session 1 if Python syntax, arrays or functions are unfamiliar, or if you are moving from MATLAB, IDL, R or Mathematica. Run cells top to bottom with **Shift+Enter** (`⏻` **Start compute** in the book toolbar, or in JupyterLab), and verify finished work with **Kernel → Restart Kernel and Run All Cells**. Predict each output before running.

## Names, containers and imports

`import numpy as np` loads the `numpy` numerical library under the short alias `np`. A Python `list` (`[...]`) uses **0-based indexing**: index `0` is first, negative index `-1` is last, and a slice `start:stop` includes `start` while **excluding `stop`** (`[:2]` returns indices `0` and `1`). A `dict` (`{key: value}`) stores named metadata, and `help(fn)` inspects unfamiliar functions.

In [ ]:
import numpy as np

measurements = [1.0, 3.0, 5.0]
observation = {'object_id': 'mock-01', 'unit': 'km/s'}
print('First and last:', measurements[0], measurements[-1], '| unit:', observation['unit'])
print('Slice [:2] (stop index 2 excluded):', measurements[:2])

## Lists vs NumPy arrays and masks

A Python `list` is a general container (`[1, 2] + [3, 4]` concatenates lists instead of adding numbers). `np.array(...)` builds a typed `np.ndarray` vector or grid — here 1D shape `(4,)` — supporting fast elementwise arithmetic and boolean masks. Missing entries use `np.nan` and are tested with `np.isfinite(...)`. Combine array masks using `&` or `|` **with parentheses around each comparison**, and use `assert` to check expectations automatically.

In [ ]:
values = np.array([1.0, 3.0, np.nan, 7.0])
keep = np.isfinite(values) & (values > 2.0)
print('shape:', values.shape, '| dtype:', values.dtype)
print('mask:', keep, '| kept:', values[keep])
assert values[keep].tolist() == [3.0, 7.0]

## Loops, vectorised arrays and functions

`for` loops iterate item by item using 4-space indentation. Whether evaluating a theoretical field on a grid or transforming survey columns, prefer vectorised NumPy array expressions over Python loops. Wrap reusable calculations in `def` functions: `return` hands back a value for code and tests, whereas `print` only displays text.

In [ ]:
def square(value):
    return value ** 2

for item in [1.0, 2.0, 3.0]:
    print(item, '->', square(item))

grid = np.array([1.0, 2.0, 3.0, 4.0])
print('Vectorised array output:', square(grid))
assert square(4.0) == 16.0

## Read a traceback

Read Python tracebacks **from the bottom line up**: check the final exception type and message first, then look up to the `---->` arrow marking the line in your code that failed.

```text
Traceback (most recent call last):
  Cell In[4], line 2
    1 fluxes = np.array([10.0, 20.0, 30.0])
----> 2 calibrated = fluxes * np.array([0.9, 1.1])
ValueError: operands could not be broadcast together with shapes (3,) (2,)
```

`ValueError` shows that 1D shapes `(3,)` and `(2,)` do not match on line 2. Check `.shape` on both arrays and supply matching lengths:

In [ ]:
fluxes = np.array([10.0, 20.0, 30.0])
bad_gains = np.array([0.9, 1.1])
try:
    fluxes * bad_gains
except ValueError as exc:
    print(f'{type(exc).__name__}: {exc}')

gains = np.array([0.9, 1.0, 1.1])
assert fluxes.shape == gains.shape == (3,)
calibrated = fluxes * gains
print('Calibrated:', calibrated)

## Views vs `.copy()` in memory

Unlike MATLAB, R or Mathematica — which copy arrays on write — a NumPy slice is a **view** into the same memory buffer, so modifying a slice mutates the original array. Call `.copy()` when you need an independent array.

![Diagram: A NumPy slice shares memory; .copy() allocates an independent array](../book/diagrams/array-views.svg)

**Shared buffer vs independent copy.** Slicing aliases `original`; `.copy()` allocates separate memory. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/array-views.dot).

In [ ]:
original = np.arange(5)
view = original[:2]
view[0] = 99
print('After view[0] = 99     -> original:', original)
assert original[0] == 99

separate = original[:2].copy()
separate[0] = -5
print('After separate[0] = -5 -> original:', original, '| separate:', separate)
assert original[0] == 99

## Your turn

1. Write `positive_finite(values)` returning a 1D array of finite entries strictly greater than zero (`> 0`).
2. Predict its output on `[-2, 0, 1, np.nan, np.inf, 3]`, then verify with `assert` *(hint: begin with `values = np.asarray(values, dtype=float)`).*
3. Explain why replacing missing measurements or failed grid points with `0.0` corrupts a scientific summary.

**Ready for Session 1:** explain an `import`, `.shape`, a parenthesised `&` mask, and `return`. Check [solutions](../book/solutions.md) once your assertion passes.

In [ ]:
# Define positive_finite(values) and check it with an assert